# 07.3 — Sentence Embeddings + FAISS

**Problem:** BERT's [CLS] is not a great sentence embedding out of the box. It was trained on NSP (predict if sentence B follows A), not on semantic similarity. Averaging token embeddings is slightly better but still weak.

**SBERT (Sentence-BERT):** Fine-tune a bi-encoder with contrastive learning on NLI/STS data. Now [CLS] (or mean-pooled) embeddings are geometrically meaningful — cosine similarity reflects semantic similarity.

**FAISS (Facebook AI Similarity Search):** Given millions of embeddings, find the top-k most similar to a query in milliseconds. We'll build a flat index from scratch, then compare to FAISS.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

# ---- Bi-encoder architecture ----
# A bi-encoder encodes query and document INDEPENDENTLY.
# Similarity = dot product or cosine of the two embeddings.
# This enables offline encoding of documents + fast online retrieval.

class MeanPoolingEncoder(nn.Module):
    """
    Sentence encoder: encode tokens, then mean-pool (ignoring padding).
    Mean pooling typically outperforms [CLS] for sentence similarity.
    """
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            self._make_layer(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
    
    def _make_layer(self, d_model, n_heads):
        return nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            activation='gelu', batch_first=True, dropout=0.1
        )
    
    def forward(self, tokens, attention_mask=None):
        """
        attention_mask: 1 for real tokens, 0 for padding.
        We exclude padding tokens from the mean.
        """
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device).unsqueeze(0)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        
        # PyTorch TransformerEncoderLayer uses src_key_padding_mask
        # True = ignore (opposite of our attention_mask convention)
        pad_mask = None
        if attention_mask is not None:
            pad_mask = (attention_mask == 0)  # True = pad position
        
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=pad_mask)
        x = self.norm(x)
        
        # Mean pool over non-padding tokens
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            x = (x * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)
        else:
            x = x.mean(1)
        
        # L2 normalize for cosine similarity via dot product
        return F.normalize(x, p=2, dim=-1)


VOCAB = 300
encoder = MeanPoolingEncoder(VOCAB)
n_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder params: {n_params:,}")

# Encode two sentences
s1 = torch.randint(1, VOCAB, (1, 12))
s2 = torch.randint(1, VOCAB, (1, 12))
e1, e2 = encoder(s1), encoder(s2)
sim = (e1 * e2).sum(-1).item()
print(f"Sentence embedding dim: {e1.shape[-1]}")
print(f"Cosine similarity (random): {sim:.4f}")
print(f"Note: L2 normalized, so dot product = cosine similarity")

In [ ]:
# ---- Contrastive Learning: SimCSE / NLI training ----
# Two contrastive objectives:
# 1. MultipleNegativesRankingLoss (MNR): given (anchor, positive), treat
#    all other positives in the batch as negatives. In-batch negatives.
# 2. Triplet loss: (anchor, positive, negative), margin-based.

class MultipleNegativesRankingLoss(nn.Module):
    """
    Given pairs (anchor, positive):
    - anchor[i] should be similar to positive[i]
    - anchor[i] should be dissimilar to positive[j] for j != i
    
    Compute similarity matrix [B x B], then cross-entropy along rows.
    This is equivalent to InfoNCE / NT-Xent loss.
    """
    def __init__(self, scale=20.0):
        super().__init__()
        self.scale = scale  # temperature scaling (high = sharper distribution)
    
    def forward(self, anchors: torch.Tensor, positives: torch.Tensor):
        # Both are L2-normalized: dot product = cosine similarity
        # sim[i, j] = similarity between anchor[i] and positive[j]
        sim = anchors @ positives.T  # [B, B]
        sim = sim * self.scale
        # Correct pair is on the diagonal
        labels = torch.arange(sim.size(0), device=sim.device)
        return F.cross_entropy(sim, labels)


class TripletLoss(nn.Module):
    """Ensure d(anchor, positive) + margin < d(anchor, negative)."""
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        pos_sim = (anchor * positive).sum(-1)  # cosine
        neg_sim = (anchor * negative).sum(-1)
        return F.relu(neg_sim - pos_sim + self.margin).mean()


# Simulate training with MNR loss
encoder = MeanPoolingEncoder(VOCAB, d_model=64)
mnr_loss = MultipleNegativesRankingLoss(scale=20.0)
optimizer = torch.optim.AdamW(encoder.parameters(), lr=2e-4)

print("Simulated contrastive training (random data):")
for step in range(100):
    # In real training: anchor = sentence, positive = paraphrase/entailment
    anchors_tok = torch.randint(1, VOCAB, (16, 12))
    positives_tok = torch.randint(1, VOCAB, (16, 12))
    
    a_emb = encoder(anchors_tok)
    p_emb = encoder(positives_tok)
    
    loss = mnr_loss(a_emb, p_emb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 25 == 0:
        print(f"  step {step:3d}: loss={loss.item():.4f}")

print("\nMNR loss lower bound: log(1/batch_size) = log(1/16) =", f"{-math.log(16):.4f}")
print("In practice with good data and model, gets close to this bound.")

In [ ]:
# ---- Flat Index: brute-force nearest neighbor search ----
# Build from scratch to understand what FAISS optimizes.

class FlatL2Index:
    """Brute-force exact nearest neighbor search. O(N*D) per query."""
    def __init__(self):
        self.vectors = None
    
    def add(self, vecs: np.ndarray):
        """vecs: [N, D] float32"""
        if self.vectors is None:
            self.vectors = vecs
        else:
            self.vectors = np.vstack([self.vectors, vecs])
    
    def search(self, query: np.ndarray, k: int):
        """Returns (distances, indices) — L2 distances, ascending."""
        # query: [1, D] or [D]
        q = query.reshape(1, -1)
        # L2^2 = ||q||^2 + ||v||^2 - 2*(q @ v^T)
        dists = np.sum((self.vectors - q) ** 2, axis=1)
        idx = np.argsort(dists)[:k]
        return dists[idx], idx


class FlatIPIndex:
    """Brute-force inner product (cosine similarity for L2-normalized vecs)."""
    def __init__(self):
        self.vectors = None
    
    def add(self, vecs: np.ndarray):
        if self.vectors is None:
            self.vectors = vecs
        else:
            self.vectors = np.vstack([self.vectors, vecs])
    
    def search(self, query: np.ndarray, k: int):
        """Returns (scores, indices) — inner products, descending."""
        q = query.reshape(1, -1)
        scores = (self.vectors @ q.T).flatten()
        idx = np.argsort(-scores)[:k]
        return scores[idx], idx


# Build a document index
D = 64  # embedding dimension
N = 1000  # number of documents

# Generate random normalized document embeddings
doc_vecs = np.random.randn(N, D).astype(np.float32)
doc_vecs /= np.linalg.norm(doc_vecs, axis=1, keepdims=True)

index = FlatIPIndex()
index.add(doc_vecs)

# Query
query_vec = np.random.randn(D).astype(np.float32)
query_vec /= np.linalg.norm(query_vec)

scores, ids = index.search(query_vec, k=5)
print("Top-5 results:")
for score, idx in zip(scores, ids):
    print(f"  doc_{idx:4d}: cosine_sim={score:.4f}")

print(f"\nFlat index: O({N}*{D}) = {N*D:,} multiplications per query")
print("With 1M docs @ 768-dim: 768M mults per query = ~0.3ms on CPU")
print("With 100M docs: too slow. Need approximate search (FAISS IVF/HNSW).")

In [ ]:
# ---- IVF: Inverted File Index (approximate, fast) ----
# Idea: partition vectors into clusters. At query time:
# 1. Find the nearest cluster centers (nprobe clusters)
# 2. Search only within those clusters
# Trade-off: speed vs recall (nprobe controls this)

class IVFIndex:
    """
    Simple IVF with k-means clustering.
    nlist = number of clusters (cells)
    nprobe = how many cells to search at query time
    """
    def __init__(self, nlist=10):
        self.nlist = nlist
        self.centroids = None
        self.cells = {i: [] for i in range(nlist)}  # cell_id -> list of (vec, global_idx)
    
    def _kmeans(self, vecs, n_clusters, n_iter=20):
        """Simple k-means."""
        # Random initialization
        idx = np.random.choice(len(vecs), n_clusters, replace=False)
        centroids = vecs[idx].copy()
        for _ in range(n_iter):
            # Assign
            dists = np.linalg.norm(vecs[:, None] - centroids[None], axis=2)  # [N, K]
            assignments = dists.argmin(1)
            # Update
            new_centroids = np.array([
                vecs[assignments == k].mean(0) if (assignments == k).any() else centroids[k]
                for k in range(n_clusters)
            ])
            if np.allclose(centroids, new_centroids): break
            centroids = new_centroids
        return centroids, assignments
    
    def train(self, vecs: np.ndarray):
        """Cluster the vectors to find centroids."""
        self.centroids, assignments = self._kmeans(vecs, self.nlist)
        for i, (vec, cell) in enumerate(zip(vecs, assignments)):
            self.cells[cell].append((vec, i))
    
    def search(self, query: np.ndarray, k: int, nprobe=3):
        """Search nprobe nearest cells, return top-k overall."""
        q = query.reshape(1, -1)
        # Find nearest centroids
        centroid_dists = np.linalg.norm(self.centroids - q, axis=1)
        probe_cells = np.argsort(centroid_dists)[:nprobe]
        
        # Gather candidates from probed cells
        candidates = []
        for cell_id in probe_cells:
            candidates.extend(self.cells[cell_id])
        
        if not candidates:
            return np.array([]), np.array([])
        
        cand_vecs = np.array([c[0] for c in candidates])
        cand_ids  = [c[1] for c in candidates]
        scores = (cand_vecs @ q.T).flatten()
        top_k_local = np.argsort(-scores)[:k]
        return scores[top_k_local], np.array(cand_ids)[top_k_local]


# Compare flat vs IVF
ivf = IVFIndex(nlist=20)
ivf.train(doc_vecs)

flat_scores, flat_ids = index.search(query_vec, k=5)
ivf_scores, ivf_ids = ivf.search(query_vec, k=5, nprobe=5)

flat_set = set(flat_ids.tolist())
ivf_set  = set(ivf_ids.tolist())
recall = len(flat_set & ivf_set) / len(flat_set)

print(f"Flat top-5 ids: {sorted(flat_ids)}")
print(f"IVF  top-5 ids: {sorted(ivf_ids)}")
print(f"Recall@5 (IVF vs Flat): {recall:.2f}")
print()
print("IVF only searches candidates in nprobe=5 out of 20 clusters.")
print("FAISS adds product quantization (PQ) for memory compression on top of IVF.")

In [ ]:
# ---- FAISS API (what you'd use in production) ----
print("""FAISS production usage:

import faiss
import numpy as np

D = 768  # embedding dimension

# Flat (exact) cosine similarity
index = faiss.IndexFlatIP(D)
faiss.normalize_L2(doc_vecs)        # L2-normalize for cosine via inner product
index.add(doc_vecs)                 # O(1) add

# Approximate (IVF + PQ) for millions of vectors
quantizer = faiss.IndexFlatIP(D)
index = faiss.IndexIVFPQ(quantizer, D, nlist=4096, m=8, nbits=8)
index.train(doc_vecs)               # k-means clustering (one-time)
index.add(doc_vecs)
index.nprobe = 64                   # how many clusters to search

# Query
faiss.normalize_L2(query_vecs)
scores, indices = index.search(query_vecs, k=10)  # returns [N_queries, k]
"""
)

print("FAISS index types:")
print("  IndexFlatL2     : exact, L2 distance, slowest, perfect recall")
print("  IndexFlatIP     : exact, inner product, use for normalized vecs")
print("  IndexIVFFlat    : approximate, fast, no compression")
print("  IndexIVFPQ      : approximate, fast, compressed (less memory)")
print("  IndexHNSW       : graph-based, fast, excellent recall, high memory")
print()
print("Rule of thumb:")
print("  < 1M vectors:  FlatIP is fine")
print("  1M-100M:       IVFPQ or HNSW")
print("  > 100M:        Distributed FAISS (faiss.swigfaiss sharded across GPUs)")

## Summary

**Bi-encoder:** Encode query and document independently → enables offline indexing and fast retrieval.

**Contrastive training:** MNR loss (in-batch negatives) pushes similar sentences together, dissimilar ones apart.

**FAISS:** Flat = exact, IVF = approximate (cluster, probe subset), HNSW = graph-based. The nprobe knob trades recall for speed.

**What breaks it:** Hard negatives. Randomly sampled negatives are too easy — the model doesn't learn fine-grained similarity. Solution: mine hard negatives using BM25 or a first-pass retriever.

---
**Next:** 07.4 — Dense Retrieval (DPR)